# Microsoft Foundry · Assistente com memória de longo prazo (Memory Store)

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aulas 21 e 22): uso do **Foundry Memory Store** para que um agente **lembre fatos de cada usuário entre sessões** (alergias, preferências, linguagem favorita…) sem montar banco de dados, banco vetorial ou lógica de embeddings.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Pré-requisitos: modelos e permissão (RBAC)](#1.-Pré-requisitos:-modelos-e-permissão-(RBAC))
2. [Instalação das bibliotecas](#2.-Instalação-das-bibliotecas)
3. [Variáveis de ambiente](#3.-Variáveis-de-ambiente)
4. [Conectando ao projeto do Foundry](#4.-Conectando-ao-projeto-do-Foundry)
5. [Criando o Memory Store](#5.-Criando-o-Memory-Store)
6. [Função para atualizar as memórias](#6.-Função-para-atualizar-as-memórias)
7. [Criando o agente](#7.-Criando-o-agente)
8. [Chat com memória](#8.-Chat-com-memória)
9. [Verificando no portal do Foundry](#9.-Verificando-no-portal-do-Foundry)
10. [Resumo dos aprendizados](#10.-Resumo-dos-aprendizados)
11. [Limpeza e custos](#11.-Limpeza-e-custos)
12. [Interface com Streamlit](#12.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Ao colocar agentes em produção, um dos maiores problemas é a **memória contextual**:

- Para lembrar algo entre sessões, era preciso montar **bancos externos** (vetoriais, NoSQL) e várias chamadas de API para gravar e recuperar.
- Quando a sessão expira, o contexto se perde.
- Reenviar **todo o histórico** ao modelo esgota tokens, gera custo e esbarra em limites de taxa.

É o que já existe em produtos como o **Microsoft 365 Copilot** (opção *Memória* nas configurações de personalização): ele lembra detalhes de conversas de semanas atrás. Como ter isso no **meu** agente?

### 0.2 Objetivo

- Criar um **Memory Store** no Foundry, que já traz embutidos o armazenamento, a vetorização e a busca.
- Gravar fatos por **usuário** (o **escopo**) a partir de mensagens livres.
- Em cada pergunta, **buscar as memórias relevantes** e entregá-las ao agente como contexto.
- Ver o agente responder de forma **personalizada** (ex.: receitas sem os alimentos a que o usuário é alérgico).

### 0.3 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Foundry Memory Store** (*preview*) | Guarda as memórias por escopo (texto + embedding) e faz a busca por similaridade. |
| **Azure OpenAI: GPT-4.1** | Modelo do agente **e** componente LLM do memory store (extrai e mescla os fatos). |
| **Azure OpenAI: text-embedding-3-small** | Gera os vetores (embeddings) das memórias e das perguntas. |
| **Foundry Agent Service** | Hospeda o `helpful-assistant-agent`. |
| **Microsoft Entra ID + RBAC** | Autenticação sem chave e a função **Azure AI User** para a identidade do projeto. |
| **Managed Identity** do projeto | Identidade usada pelo memory store para chamar os modelos. |

### 0.4 Como funciona (visão simples)

![Fluxo do Memory Store](fluxo_memory_store.svg)

### 0.5 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

### 0.6 Conceitos do Memory Store

| Conceito | Explicação |
| --- | --- |
| **Componente LLM** | Lê a mensagem e **seleciona e mescla** só os fatos úteis. "Sou alérgico a laticínios, me mostre uma receita de bolo" vira apenas "usuário é alérgico a laticínios". |
| **Componente de embedding** | Cada memória é guardada como **texto + vetor**. |
| **Busca** | A pergunta também vira vetor; as memórias mais **parecidas** (top-k, aqui 5) são devolvidas. |
| **Escopo** (`scope`) | Funciona como o **identificador** em um banco de dados: cada usuário tem suas memórias. Aqui o escopo é o nome do usuário. |

**Limites na *preview*** (informados na aula, com data de 12/12/2025):

| Item | Limite |
| --- | --- |
| Modelos suportados | Somente **Azure OpenAI** |
| Escopo | Deve ser definido **explicitamente** |
| Escopos por memory store | 100 |
| Memórias por escopo | 10.000 |
| Buscas de memória | 1.000 requisições/minuto |
| Atualizações de memória | 1.000 requisições/minuto |

> 💡 Em um estudo publicado pela Microsoft, o uso do memory store aumentou bastante a precisão em testes de memória (cerca de **3 vezes**), e modelos de melhor raciocínio (GPT-4.1 em relação ao GPT-4.1 mini) escolhem melhor **o que** guardar.

---
## 1. Pré-requisitos: modelos e permissão (RBAC)

### 1.1 Implantar os modelos

Portal do Foundry → **Modelos** → **Deploy base model**, com os valores padrão:

| Modelo | Uso | Variável |
| --- | --- | --- |
| `gpt-4.1` | Agente e componente LLM do memory store | `MODEL_DEPLOYMENT_NAME` |
| `text-embedding-3-small` | Embeddings das memórias | `TEXT_EMBEDDING_MODEL_NAME` |

### 1.2 Dar a função **Azure AI User** à identidade do projeto

O memory store chama os modelos **em nome do projeto**. Para isso, a **identidade gerenciada** do projeto (criada automaticamente, do tipo *system-assigned*) precisa da função **Azure AI User** sobre o recurso que hospeda os modelos (o recurso pai do projeto).

1. Portal do Foundry → nome do projeto → **Ver todos os projetos**: anote o **recurso pai** (conta de AI Services).
2. `portal.azure.com` → **Microsoft Foundry** → recurso pai → **Projetos** → o projeto.
3. **Resource management → Identity**: a identidade *system-assigned* já existe (anote o *Object/Principal ID*).
4. **Azure role assignments → Add role assignment**:
   - **Escopo:** *Resource group* → o grupo onde estão o recurso pai e o projeto.
   - **Função:** **Azure AI User**.
5. **Save**.

> 💡 Atribuindo no **grupo de recursos**, a permissão vale para o recurso pai automaticamente (desde que os dois estejam no mesmo grupo).

> ⚠️ Sem essa etapa o código falha com erro de **autenticação/autorização** ao criar ou atualizar memórias.

---
## 2. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: cliente do projeto (`AIProjectClient`), agentes e ferramentas. Versão *beta*, fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e enviar mensagens ao agente (Responses API). |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure sem chave no código (`DefaultAzureCredential`). |


> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam nomes de classes e métodos entre versões.

In [ ]:
# Instalando as bibliotecas no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 3. Variáveis de ambiente

As configurações ficam no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Portal do Foundry → página **Início** (*Overview*) → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo de chat. | Portal do Foundry → **Modelos + endpoints**. |
| `TEXT_EMBEDDING_MODEL_NAME` | Nome do deployment do modelo de embedding. | Portal do Foundry → **Modelos** (ex.: `text-embedding-3-small`). |

```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>
TEXT_EMBEDDING_MODEL_NAME=<text-embedding-model-name>
```

> 🔒 **Segurança:** o `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção seguinte). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
# TEXT_EMBEDDING_MODEL_NAME = modelo que transforma as memórias em vetores
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
embedding_model_name = os.getenv("TEXT_EMBEDDING_MODEL_NAME")


print("embedding_model_name:", embedding_model_name)

---
## 4. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a porta de entrada do Python para o projeto no Foundry: por ele criamos agentes, listamos conexões e obtemos o cliente OpenAI.

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: o **`DefaultAzureCredential`** reaproveita um login que já existe na máquina (Azure CLI `az login`, conta Azure do VS Code, *Managed Identity* quando roda na Azure…). Assim, **nenhuma chave fica escrita no código**: quem autentica é o **Microsoft Entra ID**.

> 💡 Para rodar localmente, basta executar `az login` antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

> Neste notebook o cliente se chama **`client`** (nos outros labs, `project_client`).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential()
)

---
## 5. Criando o Memory Store

A definição padrão (`MemoryStoreDefaultDefinition`) informa os dois componentes e as opções:

| Parâmetro | Valor | Significado |
| --- | --- | --- |
| `chat_model` | `MODEL_DEPLOYMENT_NAME` (GPT-4.1) | LLM que **extrai e mescla** os fatos de cada mensagem. |
| `embedding_model` | `TEXT_EMBEDDING_MODEL_NAME` | Gera os **vetores** para a busca por similaridade. |
| `user_profile_enabled` | `True` | Mantém um **perfil** do usuário (fatos pessoais e preferências). |
| `chat_summary_enabled` | `True` | Guarda **resumos** das conversas, de forma econômica em tokens. |

Depois, `client.memory_stores.create(...)` cria o store com o nome `demo_memory_store_v1`. A `description` é só para os desenvolvedores: não muda o comportamento.

> ⚠️ O nome precisa ser único: executar esta célula de novo com o mesmo nome tende a falhar, porque o store já existe. Para recriar, use outro nome (ex.: `_v2`) ou apague o anterior.

In [ ]:
# Definindo o Memory Store (LLM + embedding + opções) e criando no projeto
from azure.ai.projects.models import MemoryStoreDefaultDefinition, MemoryStoreDefaultOptions

# Define the Memory Store
definition = MemoryStoreDefaultDefinition(
    chat_model = model_deployment_name,
    embedding_model=embedding_model_name,
    options = MemoryStoreDefaultOptions(
        user_profile_enabled=True,
        chat_summary_enabled=True
    )
)

# Create the Memory Store
memory_store = client.memory_stores.create(
    name = "demo_memory_store_v1",
    definition = definition,
    description = "A memory store to retain user context across sessions"
)

print("Memory Store created with name:", memory_store.name)

---
## 6. Função para atualizar as memórias

`update_memory_store(memory_store_name, user_name, user_message)`:

1. Usa o **nome do usuário como escopo**: cada usuário tem suas memórias.
2. Embrulha a mensagem em um `ResponsesUserMessageItemParam` (o formato que o memory store aceita).
3. Chama `client.memory_stores.begin_update_memories(...)`, uma operação **assíncrona** (*long-running*): o LLM extrai os fatos e o modelo de embedding gera os vetores. `update_delay=0` processa imediatamente.
4. `update_poller.result()` **espera** terminar e mostra cada operação (`kind`, `memory_id`, `content`).
5. Devolve `"updated successfully"` ou, em caso de erro, `"update failed"`.

In [ ]:
# Função que envia uma mensagem ao Memory Store no escopo do usuário
# e espera (poller) o LLM extrair os fatos e o modelo gerar os embeddings
from azure.ai.projects.models import ResponsesUserMessageItemParam

def update_memory_store(memory_store_name: str, user_name: str, user_message: str):
    try:
        
        # set the scope to associate the memories with
        scope = user_name

        user_message_item = ResponsesUserMessageItemParam(
            content = user_message
        )

        update_poller = client.memory_stores.begin_update_memories(
            name = memory_store_name,
            scope = scope,
            items = [user_message_item],
            update_delay=0
        )

        # Wait for the update operation to complete
        update_result = update_poller.result()

        print(f"Updated with {len(update_result.memory_operations)} memory operations")

        for operation in update_result.memory_operations:
            print(
                f"  - Operation: {operation.kind}, Memory ID: {operation.memory_item.memory_id}, Content: {operation.memory_item.content}"
        )
        
        return "updated successfully"
    except Exception as e:
        print("Error updating memory store:", str(e))
        return "update failed"

Testando com uma mensagem que mistura vários fatos. Em português: *"Meu nome é Kuljot Singh e eu amo programar em Python. Também sou alérgico a amendoim e laticínios."*

In [ ]:
# Guardando fatos do usuário "kuljot" (escopo) a partir de uma mensagem livre
update_memory_store(
    memory_store_name = memory_store.name,
    user_name = "kuljot",
    user_message = "My name is Kuljot Singh and I love programming in Python. I am also allergic to peanuts and dairy."
)

✅ Na aula, uma única mensagem gerou **5 memórias** separadas no escopo `kuljot`: o nome do usuário, "usuário ama programar em Python", "usuário é alérgico a amendoim", "usuário é alérgico a laticínios" e um resumo combinado. Ou seja: **fatos curtos**, e não a frase inteira.

---
## 7. Criando o agente

Um agente simples, **sem ferramentas**: a memória será entregue a ele **dentro da pergunta**.

| Campo | Valor |
| --- | --- |
| `agent_name` | `helpful-assistant-agent` |
| `model` | `MODEL_DEPLOYMENT_NAME` (GPT-4.1) |
| `instructions` | *"Você é um assistente útil. Responda a todas as perguntas de forma amigável e informativa."* |

In [ ]:
# Criando um agente simples (sem ferramentas); as memórias entram pelo prompt
from azure.ai.projects.models import PromptAgentDefinition

agent_name = "helpful-assistant-agent"

agent = client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful assistant. Respond to all queries in a friendly and informative manner.",
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

---
## 8. Chat com memória

O laço pede o **nome do usuário** (o escopo, com **maiúsculas e minúsculas exatas**) e, a cada mensagem:

| Digitado | O que acontece |
| --- | --- |
| `update` | Pede um texto e grava na memória com `update_memory_store`. |
| qualquer pergunta | 1) `search_memories` busca até **5** memórias do usuário parecidas com a pergunta (`MemorySearchOptions(max_memories=5)`). 2) Cada memória é anexada como `Memory Context: …`. 3) `openai_client.responses.create` envia tudo ao agente. |
| `exit` | Encerra o laço. |

> 💡 Repare que aqui **não usamos `conversation.id`**: cada pergunta é independente. A continuidade vem do **Memory Store**, que persiste entre sessões (hoje, amanhã, semanas depois).

> ⚠️ Detalhe do laço: ao digitar `exit`, o código marca `chat = False`, mas ainda envia essa última mensagem ao agente antes de sair (o `if` de saída não interrompe a iteração).

In [ ]:
# Chat: busca as memórias do usuário, anexa como contexto e pergunta ao agente
# "update" grava uma nova memória | "exit" encerra
from azure.ai.projects.models import MemorySearchOptions

chat = True

user_name = input("Enter the user name (or type 'exit' to quit): ")

while chat:
    user_message = input("Type \"exit\" to quit the chat, \"update\" to update memory store with agent response. or ask your question.")

    if user_message.lower() == 'exit':
        chat = False
    # Only update memory store if user entered "UPDATE"
    if user_message.strip().upper() == "UPDATE":
            user_query = input("Enter something that you want to add to the memory store:")
            update_memory_store(
                memory_store_name = memory_store.name,
                user_name = user_name,
                user_message = user_query
            )
    else:
        # creating the openai client
        openai_client = client.get_openai_client()

        # searching for memories and appending to the user message as grounding context
        query_message = ResponsesUserMessageItemParam(content = user_message)

        search_response = client.memory_stores.search_memories(
            name = memory_store.name,
            scope = user_name,
            items = [query_message],
            options = MemorySearchOptions(max_memories=5)
        )
        
        # appending the found memories to the user message to create grounding context
        if len(search_response.memories) > 0:
            for memory in search_response.memories:
                print(f"Found Memory: {memory.memory_item.content}")
                user_message += f"\nMemory Context: {memory.memory_item.content}"

        response = openai_client.responses.create(
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_message
        )

        agent_response = response.output_text

        print(f"Agent: {agent_response}\n")

### 8.1 O que aconteceu na demonstração da aula

| Mensagem | Memórias encontradas | Resposta |
| --- | --- | --- |
| "Pode me indicar receitas de lanches rápidos?" | alérgico a amendoim, alérgico a laticínios | Receitas **sem amendoim e sem laticínios**: grão-de-bico assado, palitos de legumes com homus… |
| "Me dê um código para calcular o fatorial na minha linguagem favorita" | ama programar em Python | Uma função de fatorial em **Python**. |
| `update` → "Meu astro de quadrinhos favorito é o Batman" | (2 operações de memória) | Memória gravada. |
| "Me diga qualidades do meu astro de quadrinhos favorito" | favorito = Batman | Inteligência, recursos, determinação, preparo físico, compaixão, autodisciplina… |

✅ Em nenhum momento o usuário repetiu as alergias ou a linguagem favorita: o agente **lembrou**.

---
## 9. Verificando no portal do Foundry

O memory store também pode ser ligado **direto ao agente**, sem código:

1. **Agentes → `helpful-assistant-agent` → Memories** (painel à direita).
2. Selecionar `demo_memory_store_v1`, **ativar** a opção para o agente poder adicionar e alterar memórias e definir o **escopo** (ex.: `kuljot`).
3. **Save**: é criada a **versão 2** do agente.

No *playground*, perguntando "quais as qualidades do meu astro de quadrinhos favorito?" ou "a que sou alérgico?", o agente responde com Batman e amendoim/laticínios, **sem nenhum código de busca**.

| Abordagem | Vantagem | Desvantagem |
| --- | --- | --- |
| **Código (este notebook)** | Controle total de **quando** gravar e **quando** buscar. | Mais código de "encanamento". |
| **Memória ligada ao agente (portal)** | Zero código: o agente grava e busca sozinho. | Menos controle (tudo abstraído). |

---
## 10. Resumo dos aprendizados

- O **Foundry Memory Store** entrega memória de longo prazo **sem** banco vetorial próprio: armazenamento, embeddings e busca já vêm embutidos.
- Tem dois componentes: um **LLM** que extrai e mescla fatos e um **modelo de embedding** para a busca por similaridade.
- O **escopo** separa as memórias por usuário; `begin_update_memories` grava e `search_memories` busca o top-k.
- As memórias entram como **contexto** na pergunta: personalização com **poucos tokens**, em vez de reenviar todo o histórico.
- Exige a função **Azure AI User** para a identidade gerenciada do projeto (RBAC).
- Também pode ser ligado ao agente direto no portal, trocando controle por simplicidade.

---
## 11. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. Ferramentas que trazem muito texto (páginas, APIs, documentos) aumentam bastante os tokens. |
| Memory Store | Cada atualização e busca usa os modelos (LLM + embedding): cobrança **por uso** de tokens. |
| Deployments GPT-4.1 e text-embedding-3-small | Em deployments *Standard*, cobrança por token; apague-os se não forem usados em outros labs. |

Ao terminar os estudos:

- **Portal do Foundry → Agentes**: apagar as versões do agente que não serão mais usadas.
- Ao final do curso, apagar o **grupo de recursos** no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 12. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` (seção 4) e `get_openai_client()` (seção 8) |
| `atualizar_memorias()` | o mesmo `begin_update_memories` da função `update_memory_store` (seção 6) |
| `buscar_memorias()` | o mesmo `search_memories` com `max_memories=5` do chat (seção 8) |
| `perguntar()` | anexa `Memory Context` e chama `responses.create` como no chat (seção 8) |

Pontos importantes:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso os clientes ficam em cache (`@st.cache_resource`). O histórico **na tela** fica no `st.session_state`, mas a memória **de verdade** fica no Memory Store e sobrevive ao fechar o app.
- Na barra lateral escolhemos o **usuário (escopo)** e podemos **ensinar fatos** novos; abaixo de cada resposta aparecem as **memórias usadas**.
- O app usa o memory store e o agente já criados no notebook (não recria nenhum dos dois).

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import os

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MemorySearchOptions, ResponsesUserMessageItemParam
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
MEMORY_STORE_NAME = "demo_memory_store_v1"   # criado na seção 5 do notebook
AGENT_NAME = "helpful-assistant-agent"       # criado na seção 7 do notebook

st.set_page_config(page_title="Assistente com memória · Microsoft Foundry", page_icon="🧠", layout="wide")


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
@st.cache_resource
def criar_clientes():
    client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    return client, client.get_openai_client()


# ------------------------------------------------------------------
# 3. Memórias: atualizar e buscar (mesma lógica do notebook)
# ------------------------------------------------------------------
def atualizar_memorias(usuario, mensagem):
    client, _ = criar_clientes()
    poller = client.memory_stores.begin_update_memories(
        name=MEMORY_STORE_NAME,
        scope=usuario,  # cada usuário tem as suas próprias memórias
        items=[ResponsesUserMessageItemParam(content=mensagem)],
        update_delay=0,
    )
    resultado = poller.result()  # espera o LLM extrair os fatos e gerar os embeddings
    return [f"{op.kind}: {op.memory_item.content}" for op in resultado.memory_operations]


def buscar_memorias(usuario, pergunta):
    client, _ = criar_clientes()
    resposta = client.memory_stores.search_memories(
        name=MEMORY_STORE_NAME,
        scope=usuario,
        items=[ResponsesUserMessageItemParam(content=pergunta)],
        options=MemorySearchOptions(max_memories=5),
    )
    return [memoria.memory_item.content for memoria in resposta.memories]


# ------------------------------------------------------------------
# 4. Pergunta ao agente com as memórias como contexto
# ------------------------------------------------------------------
def perguntar(pergunta, memorias):
    _, openai_client = criar_clientes()
    # Igual ao notebook: cada memória vira uma linha "Memory Context" no fim da pergunta
    texto = pergunta + "".join(f"\nMemory Context: {memoria}" for memoria in memorias)
    response = openai_client.responses.create(
        extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=texto,
    )
    return response.output_text


# ------------------------------------------------------------------
# 5. Interface
# ------------------------------------------------------------------
st.title("🧠 Assistente com memória de longo prazo")
st.caption("Foundry Memory Store: o assistente lembra fatos de cada usuário entre sessões, sem banco de dados próprio.")

if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

# O histórico de tela é separado por usuário (a memória de verdade fica no Memory Store)
if "historicos" not in st.session_state:
    st.session_state.historicos = {}

with st.sidebar:
    st.header("👤 Usuário (escopo)")
    usuario = st.text_input("Nome do usuário", value="kuljot", help="Diferencia maiúsculas de minúsculas. Cada nome tem memórias próprias.").strip()

    st.divider()
    st.header("➕ Ensinar algo ao assistente")
    novo_fato = st.text_area("Fato para guardar na memória", placeholder="Ex.: Meu super-herói favorito é o Batman.")
    if st.button("💾 Guardar na memória") and novo_fato.strip() and usuario:
        with st.spinner("Extraindo fatos e gerando embeddings..."):
            try:
                operacoes = atualizar_memorias(usuario, novo_fato)
                st.success(f"{len(operacoes)} operação(ões) de memória")
                for operacao in operacoes:
                    st.caption(f"• {operacao}")
            except (HttpResponseError, ClientAuthenticationError) as erro:
                st.error(f"Erro ao atualizar a memória: {erro}")

    st.divider()
    st.caption(f"Memory Store: `{MEMORY_STORE_NAME}`  \nAgente: `{AGENT_NAME}`")

if not usuario:
    st.info("Informe o nome do usuário na barra lateral.")
    st.stop()

historico = st.session_state.historicos.setdefault(usuario, [])
for message in historico:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if message.get("memorias"):
            with st.expander(f"🧠 Memórias usadas ({len(message['memorias'])})"):
                for memoria in message["memorias"]:
                    st.markdown(f"- {memoria}")

if prompt := st.chat_input(f"Pergunte algo como {usuario}"):
    historico.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Buscando memórias e respondendo..."):
            try:
                memorias = buscar_memorias(usuario, prompt)
                resposta = perguntar(prompt, memorias)
            except ClientAuthenticationError:
                st.error("Falha no login da Azure. Execute `az login` no terminal.")
            except (HttpResponseError, APIError) as erro:
                st.error(f"Erro ao consultar o Foundry: {erro}")
            else:
                st.markdown(resposta)
                if memorias:
                    with st.expander(f"🧠 Memórias usadas ({len(memorias)})"):
                        for memoria in memorias:
                            st.markdown(f"- {memoria}")
                historico.append({"role": "assistant", "content": resposta, "memorias": memorias})
